# Hands-on — Recommendations API

Demonstração interativa das rotas da API de personalização.

**Setup automático:** a primeira célula de código resolve `API_BASE_URL` e `API_KEY` via `terraform output` / SSM (mesmo fluxo de `testing_endpoint.ipynb`). Basta ter infra deployada e credenciais AWS configuradas na máquina.

Variáveis opcionais (sobrescrevem o auto-resolve):

```bash
export RECOMMENDATIONS_API_BASE_URL="https://<api-id>.execute-api.us-east-1.amazonaws.com/v1"
export RECOMMENDATIONS_API_KEY="<sua-api-key>"
export RECOMMENDATIONS_TEST_USER_ID="u_0231"
export RECOMMENDATIONS_TEST_COLD_START_USER_ID="u_9999"
```

## Setup — imports, URL e API key

In [1]:
import json
import os
import subprocess
import sys
import time
from pathlib import Path

import httpx
import pandas as pd

PROJECT_ROOT = (Path("..") if Path("..").joinpath("terraform").is_dir() else Path(".")).resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
TERRAFORM_DIR = PROJECT_ROOT / "terraform"
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
API_STAGE = "v1"
KNOWN_USER_ID = os.getenv("RECOMMENDATIONS_TEST_USER_ID", "u_0231")
COLD_START_USER_ID = os.getenv("RECOMMENDATIONS_TEST_COLD_START_USER_ID", "u_9999")


def _terraform_output(name: str) -> str | None:
    try:
        return subprocess.check_output(
            ["terraform", f"-chdir={TERRAFORM_DIR}", "output", "-raw", name],
            text=True,
            stderr=subprocess.DEVNULL,
        ).strip()
    except (subprocess.CalledProcessError, FileNotFoundError):
        return None


def normalize_api_base_url(base_url: str) -> str:
    normalized = base_url.rstrip("/")
    if normalized.endswith(f"/{API_STAGE}"):
        return normalized
    if "execute-api" in normalized:
        return f"{normalized}/{API_STAGE}"
    return normalized


def load_api_config() -> tuple[str, str]:
    base_url = os.getenv("RECOMMENDATIONS_API_BASE_URL")
    api_key = os.getenv("RECOMMENDATIONS_API_KEY")

    if not base_url:
        base_url = _terraform_output("recommendations_api_gateway_endpoint")

    if not api_key:
        api_key = _terraform_output("recommendations_api_key")

    if not api_key:
        param_name = _terraform_output("recommendations_api_key_ssm_parameter")
        if param_name:
            import boto3

            ssm = boto3.client("ssm", region_name=AWS_REGION)
            api_key = ssm.get_parameter(Name=param_name, WithDecryption=True)[
                "Parameter"
            ]["Value"]

    if not base_url or not api_key:
        raise RuntimeError(
            "Defina RECOMMENDATIONS_API_BASE_URL e RECOMMENDATIONS_API_KEY "
            "ou aplique o Terraform e configure credenciais AWS."
        )

    return normalize_api_base_url(base_url), api_key


def api_headers(*, with_key: bool = True) -> dict[str, str]:
    headers = {"Accept": "application/json"}
    if with_key:
        headers["x-api-key"] = API_KEY
    return headers


def call_api(
    method: str,
    path: str,
    *,
    json_body: dict | None = None,
    with_key: bool = True,
    timeout: float = 30.0,
) -> httpx.Response:
    url = f"{API_BASE_URL}{path}"
    with httpx.Client(timeout=timeout) as client:
        return client.request(
            method,
            url,
            headers=api_headers(with_key=with_key),
            json=json_body,
        )


def show_response(response: httpx.Response, *, label: str = "") -> httpx.Response:
    prefix = f"[{label}] " if label else ""
    print(f"{prefix}{response.request.method} {response.request.url}")
    print(f"{prefix}HTTP {response.status_code}")
    content_type = response.headers.get("content-type", "")
    if "json" in content_type:
        print(json.dumps(response.json(), indent=2, ensure_ascii=False))
    else:
        print(response.text)
    return response


def wait_for_api_health(
    *,
    timeout_seconds: float = 300,
    poll_interval_seconds: float = 10,
) -> None:
    deadline = time.monotonic() + timeout_seconds
    last_status: int | str = "unknown"
    while time.monotonic() < deadline:
        try:
            response = call_api("GET", "/health", with_key=False, timeout=10.0)
            last_status = response.status_code
            if response.status_code == 200:
                return
        except httpx.HTTPError:
            last_status = "connection_error"
        time.sleep(poll_interval_seconds)
    raise TimeoutError(
        f"/health did not return 200 within {timeout_seconds}s (last_status={last_status})"
    )


def warn_if_predictions_table_empty() -> None:
    try:
        from tests.helpers.aws_integration import (
            dynamodb_table_has_items,
            load_terraform_outputs,
        )

        outputs = load_terraform_outputs()
        table_name = outputs.get("predictions_dynamodb_table_name")
        if table_name and not dynamodb_table_has_items(table_name):
            print(
                f"AVISO: tabela DynamoDB '{table_name}' vazia. "
                "Rode model_predict antes dos testes de usuário conhecido."
            )
    except Exception as error:  # noqa: BLE001
        print(f"AVISO: não foi possível verificar DynamoDB ({error}).")


API_BASE_URL, API_KEY = load_api_config()
wait_for_api_health()
warn_if_predictions_table_empty()

print(f"API base URL: {API_BASE_URL}")
print(f"Known user:   {KNOWN_USER_ID}")
print(f"Cold start:   {COLD_START_USER_ID}")

AVISO: não foi possível verificar DynamoDB (No module named 'pytest').
API base URL: https://2ui1x3o7wb.execute-api.us-east-1.amazonaws.com/v1
Known user:   u_0231
Cold start:   u_9999


## Leitura de dados necessários

In [2]:
events = pd.read_csv(DATA_DIR / "events.csv")
products = pd.read_csv(DATA_DIR / "products.csv")

print(f"events:   {len(events):,} linhas · {events['user_id'].nunique()} usuários")
print(f"products: {len(products):,} produtos · categorias: {sorted(products['category'].unique())}")

sample_user = KNOWN_USER_ID
user_events = events[events["user_id"] == sample_user]
print(f"\nUsuário demo {sample_user}: {len(user_events)} eventos")
display(user_events.head(3))

events:   8,000 linhas · 500 usuários
products: 60 produtos · categorias: ['beleza', 'casa', 'eletronicos', 'esporte', 'livros', 'moda']

Usuário demo u_0231: 18 eventos


,user_id,product_id,event_type,timestamp
0,u_0231,p_042,view,2026-01-01 00:09:00
423,u_0231,p_032,view,2026-01-09 06:36:00
912,u_0231,p_055,purchase,2026-01-17 16:39:00


## Requisições por rota

### `GET /health`

In [3]:
show_response(call_api("GET", "/health", with_key=False), label="health")

[health] GET https://2ui1x3o7wb.execute-api.us-east-1.amazonaws.com/v1/health
[health] HTTP 200
{
  "status": "ok"
}


<Response [200 OK]>

### `GET /recommendations/{user_id}`

In [4]:
print("--- usuário com histórico ---")
show_response(
    call_api("GET", f"/recommendations/{KNOWN_USER_ID}"),
    label="recommendations",
)

print("\n--- cold start ---")
show_response(
    call_api("GET", f"/recommendations/{COLD_START_USER_ID}"),
    label="cold_start",
)

--- usuário com histórico ---
[recommendations] GET https://2ui1x3o7wb.execute-api.us-east-1.amazonaws.com/v1/recommendations/u_0231
[recommendations] HTTP 200
{
  "user_id": "u_0231",
  "cold_start_flag": false,
  "count": 10,
  "recommendations": [
    {
      "product_id": "p_032",
      "score": 0.206472161039946
    },
    {
      "product_id": "p_026",
      "score": 0.2034872902495403
    },
    {
      "product_id": "p_057",
      "score": 0.13124761043915797
    },
    {
      "product_id": "p_059",
      "score": 0.09371419024752797
    },
    {
      "product_id": "p_039",
      "score": 0.09328103329528584
    },
    {
      "product_id": "p_058",
      "score": 0.0903509648974111
    },
    {
      "product_id": "p_000",
      "score": 0.08241557144476351
    },
    {
      "product_id": "p_015",
      "score": 0.08206454678397163
    },
    {
      "product_id": "p_042",
      "score": 0.08138293906541968
    },
    {
      "product_id": "p_022",
      "score": 0.07568379

<Response [200 OK]>

### `GET /metrics` — Prometheus (default)

In [5]:
show_response(call_api("GET", "/metrics"), label="metrics_prometheus")

[metrics_prometheus] GET https://2ui1x3o7wb.execute-api.us-east-1.amazonaws.com/v1/metrics
[metrics_prometheus] HTTP 200
# HELP recommendations_api_requests_total Total HTTP requests handled.
# TYPE recommendations_api_requests_total counter
recommendations_api_requests_total 9.0
# HELP recommendations_api_errors_total Total HTTP errors.
# TYPE recommendations_api_errors_total counter
recommendations_api_errors_total 1.0
# HELP recommendations_api_cold_start_total Total cold-start fallbacks.
# TYPE recommendations_api_cold_start_total counter
recommendations_api_cold_start_total 2.0
# HELP recommendations_api_latency_ms Request latency in milliseconds.
# TYPE recommendations_api_latency_ms summary
recommendations_api_latency_ms_count 9.0
recommendations_api_latency_ms_sum 638.6757299999317
recommendations_api_latency_ms{quantile="0.5"} 19.16752799999699
recommendations_api_latency_ms{quantile="0.95"} 311.6751378000002
# HELP recommendations_api_latency_avg_ms Average request latency.
#

<Response [200 OK]>

### `GET /metrics?format=datadog`

In [6]:
show_response(call_api("GET", "/metrics?format=datadog"), label="metrics_datadog")

[metrics_datadog] GET https://2ui1x3o7wb.execute-api.us-east-1.amazonaws.com/v1/metrics?format=datadog
[metrics_datadog] HTTP 200
{
  "series": [
    {
      "metric": "recommendations_api.requests.total",
      "type": 1,
      "points": [
        {
          "timestamp": 1785598531,
          "value": 9.0
        }
      ],
      "tags": [
        "service:recommendations_api"
      ]
    },
    {
      "metric": "recommendations_api.errors.total",
      "type": 1,
      "points": [
        {
          "timestamp": 1785598531,
          "value": 1.0
        }
      ],
      "tags": [
        "service:recommendations_api"
      ]
    },
    {
      "metric": "recommendations_api.cold_start.total",
      "type": 1,
      "points": [
        {
          "timestamp": 1785598531,
          "value": 2.0
        }
      ],
      "tags": [
        "service:recommendations_api"
      ]
    },
    {
      "metric": "recommendations_api.latency.count",
      "type": 3,
      "points": [
       

<Response [200 OK]>

### `POST /recommendations_filtered` — exemplo básico

In [7]:
show_response(
    call_api(
        "POST",
        "/recommendations_filtered",
        json_body={"user_id": KNOWN_USER_ID, "limit": 5},
    ),
    label="filtered_basic",
)

[filtered_basic] POST https://2ui1x3o7wb.execute-api.us-east-1.amazonaws.com/v1/recommendations_filtered
[filtered_basic] HTTP 200
{
  "user_id": "u_0231",
  "cold_start_flag": false,
  "count": 5,
  "category": null,
  "recommendations": [
    {
      "user_id": "u_0231",
      "product_id": "p_032",
      "is_cold_start": false,
      "interactions": 3,
      "price": 24.31,
      "avg_rating": 3.2,
      "popularity_score": 0.125,
      "user_affinity_match": 1,
      "recommendation_score": 0.206472161039946,
      "category": "esporte"
    },
    {
      "user_id": "u_0231",
      "product_id": "p_026",
      "is_cold_start": false,
      "interactions": 3,
      "price": 129.92,
      "avg_rating": 4.6,
      "popularity_score": 0.292,
      "user_affinity_match": 1,
      "recommendation_score": 0.2034872902495403,
      "category": "esporte"
    },
    {
      "user_id": "u_0231",
      "product_id": "p_057",
      "is_cold_start": false,
      "interactions": 2,
      "price":

<Response [200 OK]>

### `POST /recommendations_filtered` — todos os filtros

Cada célula abaixo demonstra um filtro (ou par de filtros) do contrato da API.

In [8]:
baseline = call_api("GET", f"/recommendations/{KNOWN_USER_ID}").json()
sample_product_id = baseline["recommendations"][0]["product_id"]
sample_category = products.loc[products["product_id"] == sample_product_id, "category"].iloc[0]
price_stats = products["price"]
min_price_demo = float(price_stats.quantile(0.25))
max_price_demo = float(price_stats.quantile(0.75))

print(f"Produto de referência: {sample_product_id} · categoria: {sample_category}")
print(f"Faixa de preço demo: {min_price_demo:.2f} – {max_price_demo:.2f}")

Produto de referência: p_032 · categoria: esporte
Faixa de preço demo: 174.47 – 615.39


#### `limit`

In [9]:
show_response(
    call_api(
        "POST",
        "/recommendations_filtered",
        json_body={"user_id": KNOWN_USER_ID, "limit": 3},
    ),
    label="filter_limit",
)

[filter_limit] POST https://2ui1x3o7wb.execute-api.us-east-1.amazonaws.com/v1/recommendations_filtered
[filter_limit] HTTP 200
{
  "user_id": "u_0231",
  "cold_start_flag": false,
  "count": 3,
  "category": null,
  "recommendations": [
    {
      "user_id": "u_0231",
      "product_id": "p_032",
      "is_cold_start": false,
      "interactions": 3,
      "price": 24.31,
      "avg_rating": 3.2,
      "popularity_score": 0.125,
      "user_affinity_match": 1,
      "recommendation_score": 0.206472161039946,
      "category": "esporte"
    },
    {
      "user_id": "u_0231",
      "product_id": "p_026",
      "is_cold_start": false,
      "interactions": 3,
      "price": 129.92,
      "avg_rating": 4.6,
      "popularity_score": 0.292,
      "user_affinity_match": 1,
      "recommendation_score": 0.2034872902495403,
      "category": "esporte"
    },
    {
      "user_id": "u_0231",
      "product_id": "p_057",
      "is_cold_start": false,
      "interactions": 2,
      "price": 353

<Response [200 OK]>

#### `exclude_product_ids`

In [10]:
show_response(
    call_api(
        "POST",
        "/recommendations_filtered",
        json_body={
            "user_id": KNOWN_USER_ID,
            "limit": 5,
            "exclude_product_ids": [sample_product_id],
        },
    ),
    label="filter_exclude_product_ids",
)

[filter_exclude_product_ids] POST https://2ui1x3o7wb.execute-api.us-east-1.amazonaws.com/v1/recommendations_filtered
[filter_exclude_product_ids] HTTP 200
{
  "user_id": "u_0231",
  "cold_start_flag": false,
  "count": 5,
  "category": null,
  "recommendations": [
    {
      "user_id": "u_0231",
      "product_id": "p_026",
      "is_cold_start": false,
      "interactions": 3,
      "price": 129.92,
      "avg_rating": 4.6,
      "popularity_score": 0.292,
      "user_affinity_match": 1,
      "recommendation_score": 0.2034872902495403,
      "category": "esporte"
    },
    {
      "user_id": "u_0231",
      "product_id": "p_057",
      "is_cold_start": false,
      "interactions": 2,
      "price": 353.48,
      "avg_rating": 4.5,
      "popularity_score": 0.326,
      "user_affinity_match": 1,
      "recommendation_score": 0.13124761043915797,
      "category": "esporte"
    },
    {
      "user_id": "u_0231",
      "product_id": "p_059",
      "is_cold_start": false,
      "inter

<Response [200 OK]>

#### `category`

In [11]:
show_response(
    call_api(
        "POST",
        "/recommendations_filtered",
        json_body={
            "user_id": KNOWN_USER_ID,
            "limit": 5,
            "category": sample_category,
        },
    ),
    label="filter_category",
)

[filter_category] POST https://2ui1x3o7wb.execute-api.us-east-1.amazonaws.com/v1/recommendations_filtered
[filter_category] HTTP 200
{
  "user_id": "u_0231",
  "cold_start_flag": false,
  "count": 5,
  "category": "esporte",
  "recommendations": [
    {
      "user_id": "u_0231",
      "product_id": "p_032",
      "is_cold_start": false,
      "interactions": 3,
      "price": 24.31,
      "avg_rating": 3.2,
      "popularity_score": 0.125,
      "user_affinity_match": 1,
      "recommendation_score": 0.206472161039946,
      "category": "esporte"
    },
    {
      "user_id": "u_0231",
      "product_id": "p_026",
      "is_cold_start": false,
      "interactions": 3,
      "price": 129.92,
      "avg_rating": 4.6,
      "popularity_score": 0.292,
      "user_affinity_match": 1,
      "recommendation_score": 0.2034872902495403,
      "category": "esporte"
    },
    {
      "user_id": "u_0231",
      "product_id": "p_057",
      "is_cold_start": false,
      "interactions": 2,
      "

<Response [200 OK]>

#### `categories` (whitelist com várias categorias)

In [12]:
show_response(
    call_api(
        "POST",
        "/recommendations_filtered",
        json_body={
            "user_id": KNOWN_USER_ID,
            "limit": 5,
            "categories": ["esporte", "moda"],
        },
    ),
    label="filter_categories",
)

[filter_categories] POST https://2ui1x3o7wb.execute-api.us-east-1.amazonaws.com/v1/recommendations_filtered
[filter_categories] HTTP 200
{
  "user_id": "u_0231",
  "cold_start_flag": false,
  "count": 5,
  "category": null,
  "recommendations": [
    {
      "user_id": "u_0231",
      "product_id": "p_032",
      "is_cold_start": false,
      "interactions": 3,
      "price": 24.31,
      "avg_rating": 3.2,
      "popularity_score": 0.125,
      "user_affinity_match": 1,
      "recommendation_score": 0.206472161039946,
      "category": "esporte"
    },
    {
      "user_id": "u_0231",
      "product_id": "p_026",
      "is_cold_start": false,
      "interactions": 3,
      "price": 129.92,
      "avg_rating": 4.6,
      "popularity_score": 0.292,
      "user_affinity_match": 1,
      "recommendation_score": 0.2034872902495403,
      "category": "esporte"
    },
    {
      "user_id": "u_0231",
      "product_id": "p_057",
      "is_cold_start": false,
      "interactions": 2,
      "p

<Response [200 OK]>

#### `exclude_categories`

In [13]:
show_response(
    call_api(
        "POST",
        "/recommendations_filtered",
        json_body={
            "user_id": KNOWN_USER_ID,
            "limit": 5,
            "exclude_categories": ["livros"],
        },
    ),
    label="filter_exclude_categories",
)

[filter_exclude_categories] POST https://2ui1x3o7wb.execute-api.us-east-1.amazonaws.com/v1/recommendations_filtered
[filter_exclude_categories] HTTP 200
{
  "user_id": "u_0231",
  "cold_start_flag": false,
  "count": 5,
  "category": null,
  "recommendations": [
    {
      "user_id": "u_0231",
      "product_id": "p_032",
      "is_cold_start": false,
      "interactions": 3,
      "price": 24.31,
      "avg_rating": 3.2,
      "popularity_score": 0.125,
      "user_affinity_match": 1,
      "recommendation_score": 0.206472161039946,
      "category": "esporte"
    },
    {
      "user_id": "u_0231",
      "product_id": "p_026",
      "is_cold_start": false,
      "interactions": 3,
      "price": 129.92,
      "avg_rating": 4.6,
      "popularity_score": 0.292,
      "user_affinity_match": 1,
      "recommendation_score": 0.2034872902495403,
      "category": "esporte"
    },
    {
      "user_id": "u_0231",
      "product_id": "p_057",
      "is_cold_start": false,
      "interactio

<Response [200 OK]>

#### `min_price` / `max_price`

In [14]:
show_response(
    call_api(
        "POST",
        "/recommendations_filtered",
        json_body={
            "user_id": KNOWN_USER_ID,
            "limit": 5,
            "min_price": min_price_demo,
            "max_price": max_price_demo,
        },
    ),
    label="filter_price_range",
)

[filter_price_range] POST https://2ui1x3o7wb.execute-api.us-east-1.amazonaws.com/v1/recommendations_filtered
[filter_price_range] HTTP 200
{
  "user_id": "u_0231",
  "cold_start_flag": false,
  "count": 5,
  "category": null,
  "recommendations": [
    {
      "user_id": "u_0231",
      "product_id": "p_057",
      "is_cold_start": false,
      "interactions": 2,
      "price": 353.48,
      "avg_rating": 4.5,
      "popularity_score": 0.326,
      "user_affinity_match": 1,
      "recommendation_score": 0.13124761043915797,
      "category": "esporte"
    },
    {
      "user_id": "u_0231",
      "product_id": "p_042",
      "is_cold_start": false,
      "interactions": 1,
      "price": 278.1,
      "avg_rating": 4.0,
      "popularity_score": 0.068,
      "user_affinity_match": 0,
      "recommendation_score": 0.08138293906541968,
      "category": "livros"
    },
    {
      "user_id": "u_0231",
      "product_id": "p_055",
      "is_cold_start": false,
      "interactions": 1,
    

<Response [200 OK]>

#### `min_avg_rating`

In [15]:
show_response(
    call_api(
        "POST",
        "/recommendations_filtered",
        json_body={
            "user_id": KNOWN_USER_ID,
            "limit": 5,
            "min_avg_rating": 4.0,
        },
    ),
    label="filter_min_avg_rating",
)

[filter_min_avg_rating] POST https://2ui1x3o7wb.execute-api.us-east-1.amazonaws.com/v1/recommendations_filtered
[filter_min_avg_rating] HTTP 200
{
  "user_id": "u_0231",
  "cold_start_flag": false,
  "count": 5,
  "category": null,
  "recommendations": [
    {
      "user_id": "u_0231",
      "product_id": "p_026",
      "is_cold_start": false,
      "interactions": 3,
      "price": 129.92,
      "avg_rating": 4.6,
      "popularity_score": 0.292,
      "user_affinity_match": 1,
      "recommendation_score": 0.2034872902495403,
      "category": "esporte"
    },
    {
      "user_id": "u_0231",
      "product_id": "p_057",
      "is_cold_start": false,
      "interactions": 2,
      "price": 353.48,
      "avg_rating": 4.5,
      "popularity_score": 0.326,
      "user_affinity_match": 1,
      "recommendation_score": 0.13124761043915797,
      "category": "esporte"
    },
    {
      "user_id": "u_0231",
      "product_id": "p_059",
      "is_cold_start": false,
      "interactions": 

<Response [200 OK]>

#### `min_popularity_score`

In [16]:
show_response(
    call_api(
        "POST",
        "/recommendations_filtered",
        json_body={
            "user_id": KNOWN_USER_ID,
            "limit": 5,
            "min_popularity_score": 0.3,
        },
    ),
    label="filter_min_popularity_score",
)

[filter_min_popularity_score] POST https://2ui1x3o7wb.execute-api.us-east-1.amazonaws.com/v1/recommendations_filtered
[filter_min_popularity_score] HTTP 200
{
  "user_id": "u_0231",
  "cold_start_flag": false,
  "count": 5,
  "category": null,
  "recommendations": [
    {
      "user_id": "u_0231",
      "product_id": "p_057",
      "is_cold_start": false,
      "interactions": 2,
      "price": 353.48,
      "avg_rating": 4.5,
      "popularity_score": 0.326,
      "user_affinity_match": 1,
      "recommendation_score": 0.13124761043915797,
      "category": "esporte"
    },
    {
      "user_id": "u_0231",
      "product_id": "p_059",
      "is_cold_start": false,
      "interactions": 1,
      "price": 104.16,
      "avg_rating": 4.5,
      "popularity_score": 0.386,
      "user_affinity_match": 1,
      "recommendation_score": 0.09371419024752797,
      "category": "esporte"
    },
    {
      "user_id": "u_0231",
      "product_id": "p_039",
      "is_cold_start": false,
      "in

<Response [200 OK]>

#### `min_recommendation_score`

In [17]:
show_response(
    call_api(
        "POST",
        "/recommendations_filtered",
        json_body={
            "user_id": KNOWN_USER_ID,
            "limit": 5,
            "min_recommendation_score": 0.1,
        },
    ),
    label="filter_min_recommendation_score",
)

[filter_min_recommendation_score] POST https://2ui1x3o7wb.execute-api.us-east-1.amazonaws.com/v1/recommendations_filtered
[filter_min_recommendation_score] HTTP 200
{
  "user_id": "u_0231",
  "cold_start_flag": false,
  "count": 3,
  "category": null,
  "recommendations": [
    {
      "user_id": "u_0231",
      "product_id": "p_032",
      "is_cold_start": false,
      "interactions": 3,
      "price": 24.31,
      "avg_rating": 3.2,
      "popularity_score": 0.125,
      "user_affinity_match": 1,
      "recommendation_score": 0.206472161039946,
      "category": "esporte"
    },
    {
      "user_id": "u_0231",
      "product_id": "p_026",
      "is_cold_start": false,
      "interactions": 3,
      "price": 129.92,
      "avg_rating": 4.6,
      "popularity_score": 0.292,
      "user_affinity_match": 1,
      "recommendation_score": 0.2034872902495403,
      "category": "esporte"
    },
    {
      "user_id": "u_0231",
      "product_id": "p_057",
      "is_cold_start": false,
     

<Response [200 OK]>

#### `only_affinity_match`

In [18]:
show_response(
    call_api(
        "POST",
        "/recommendations_filtered",
        json_body={
            "user_id": KNOWN_USER_ID,
            "limit": 5,
            "only_affinity_match": True,
        },
    ),
    label="filter_only_affinity_match",
)

[filter_only_affinity_match] POST https://2ui1x3o7wb.execute-api.us-east-1.amazonaws.com/v1/recommendations_filtered
[filter_only_affinity_match] HTTP 200
{
  "user_id": "u_0231",
  "cold_start_flag": false,
  "count": 5,
  "category": null,
  "recommendations": [
    {
      "user_id": "u_0231",
      "product_id": "p_032",
      "is_cold_start": false,
      "interactions": 3,
      "price": 24.31,
      "avg_rating": 3.2,
      "popularity_score": 0.125,
      "user_affinity_match": 1,
      "recommendation_score": 0.206472161039946,
      "category": "esporte"
    },
    {
      "user_id": "u_0231",
      "product_id": "p_026",
      "is_cold_start": false,
      "interactions": 3,
      "price": 129.92,
      "avg_rating": 4.6,
      "popularity_score": 0.292,
      "user_affinity_match": 1,
      "recommendation_score": 0.2034872902495403,
      "category": "esporte"
    },
    {
      "user_id": "u_0231",
      "product_id": "p_057",
      "is_cold_start": false,
      "interact

<Response [200 OK]>

#### `exclude_cold_start` (usuário cold start para comparar)

In [19]:
print("--- com cold start (default) ---")
show_response(
    call_api(
        "POST",
        "/recommendations_filtered",
        json_body={"user_id": COLD_START_USER_ID, "limit": 3},
    ),
    label="cold_start_with_fallback",
)

print("\n--- exclude_cold_start=true (pode retornar vazio) ---")
show_response(
    call_api(
        "POST",
        "/recommendations_filtered",
        json_body={
            "user_id": COLD_START_USER_ID,
            "limit": 3,
            "exclude_cold_start": True,
        },
    ),
    label="filter_exclude_cold_start",
)

--- com cold start (default) ---
[cold_start_with_fallback] POST https://2ui1x3o7wb.execute-api.us-east-1.amazonaws.com/v1/recommendations_filtered
[cold_start_with_fallback] HTTP 200
{
  "user_id": "u_9999",
  "cold_start_flag": true,
  "count": 3,
  "category": null,
  "recommendations": [
    {
      "user_id": "cold_start",
      "product_id": "p_030",
      "is_cold_start": true,
      "interactions": 0,
      "price": 622.35,
      "avg_rating": 3.8,
      "popularity_score": 0.801,
      "user_affinity_match": 0,
      "recommendation_score": 0.801,
      "category": "beleza"
    },
    {
      "user_id": "cold_start",
      "product_id": "p_000",
      "is_cold_start": true,
      "interactions": 0,
      "price": 115.19,
      "avg_rating": 3.1,
      "popularity_score": 0.652,
      "user_affinity_match": 0,
      "recommendation_score": 0.652,
      "category": "esporte"
    },
    {
      "user_id": "cold_start",
      "product_id": "p_005",
      "is_cold_start": true,
   

<Response [200 OK]>

#### Combinação de filtros

In [20]:
show_response(
    call_api(
        "POST",
        "/recommendations_filtered",
        json_body={
            "user_id": KNOWN_USER_ID,
            "limit": 5,
            "exclude_product_ids": [sample_product_id],
            "category": "esporte",
            "min_price": 50,
            "max_price": 500,
            "min_recommendation_score": 0.05,
            "only_affinity_match": True,
        },
    ),
    label="filter_combined",
)

[filter_combined] POST https://2ui1x3o7wb.execute-api.us-east-1.amazonaws.com/v1/recommendations_filtered
[filter_combined] HTTP 200
{
  "user_id": "u_0231",
  "cold_start_flag": false,
  "count": 5,
  "category": "esporte",
  "recommendations": [
    {
      "user_id": "u_0231",
      "product_id": "p_026",
      "is_cold_start": false,
      "interactions": 3,
      "price": 129.92,
      "avg_rating": 4.6,
      "popularity_score": 0.292,
      "user_affinity_match": 1,
      "recommendation_score": 0.2034872902495403,
      "category": "esporte"
    },
    {
      "user_id": "u_0231",
      "product_id": "p_057",
      "is_cold_start": false,
      "interactions": 2,
      "price": 353.48,
      "avg_rating": 4.5,
      "popularity_score": 0.326,
      "user_affinity_match": 1,
      "recommendation_score": 0.13124761043915797,
      "category": "esporte"
    },
    {
      "user_id": "u_0231",
      "product_id": "p_059",
      "is_cold_start": false,
      "interactions": 1,
    

<Response [200 OK]>